# Unit 2 code companion: A Map of Models

Fit the same data twice, once with a model that names a distribution and once with one that does not, and see exactly which questions each one can answer.

Run the cells in order. Each section matches a moment in the slides. The point is to see the mechanism move when you change the inputs, so change them.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_score

rng = np.random.default_rng(220)
plt.rcParams["figure.figsize"] = (7, 3.5)

cars = pd.read_csv("https://richardson.byu.edu/220/cars.csv").dropna()
print(cars.shape)
cars.head(3)

(392, 9)


,mpg,cylinder,displacement,horsepower,weight,acceleration,model_year,origin,car_name
0,18.0,8,307.0,130,3504,12.0,70,American,chevrolet
1,15.0,8,350.0,165,3693,11.5,70,American,buick
2,18.0,8,318.0,150,3436,11.0,70,American,plymouth


## 1. The same data, two kinds of model

One names a distribution for mpg. The other just predicts it.

In [2]:
X = cars[["weight", "horsepower", "model_year"]]
y = cars["mpg"]

lin = smf.ols("mpg ~ weight + horsepower + model_year", data=cars).fit()
forest = RandomForestRegressor(n_estimators=400, random_state=0).fit(X, y)

print("linear regression, in-sample R^2:", round(lin.rsquared, 3))
print("random forest,     in-sample R^2:", round(forest.score(X, y), 3))

linear regression, in-sample R^2: 0.808
random forest,     in-sample R^2: 0.981


The forest fits better. That is not the interesting part. What each one will *tell you* is the interesting part.

## 2. What the probability model hands you



In [3]:
print(lin.summary().tables[1])
print()
print("AIC:", round(lin.aic, 1))
print("95% CI for the weight coefficient:")
print(lin.conf_int().loc["weight"].round(5).to_dict())

                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    -13.7194      4.182     -3.281      0.001     -21.941      -5.498
weight        -0.0064      0.000    -15.768      0.000      -0.007      -0.006
horsepower    -0.0050      0.009     -0.530      0.597      -0.024       0.014
model_year     0.7487      0.052     14.365      0.000       0.646       0.851

AIC: 2082.8
95% CI for the weight coefficient:
{0: -0.00725, 1: -0.00564}


A slope, a standard error, a $p$-value, a confidence interval, and an AIC. Every one of those comes out of the sentence `mpg ~ Normal(b0 + b1*weight + ..., sigma^2)`.

## 3. What the algorithmic model hands you



In [4]:
imp = pd.Series(forest.feature_importances_, index=X.columns).sort_values(ascending=False)
print(imp.round(3).to_string())

for attr in ["pvalues", "conf_int", "aic", "bse"]:
    print(f"forest.{attr}:", "yes" if hasattr(forest, attr) else "does not exist")

weight        0.676
horsepower    0.180
model_year    0.145
forest.pvalues: does not exist
forest.conf_int: does not exist
forest.aic: does not exist
forest.bse: does not exist


Importances, and nothing else. There is no likelihood, so there is no standard error, no $p$-value, and no AIC. Those quantities are not hidden, they are undefined.

## 4. Importances are not effects

The regression says weight is negative. Ask the forest for a direction and it has none.

In [5]:
print("regression coefficient on weight:", round(lin.params['weight'], 5))
print("forest importance for weight:      ", round(imp['weight'], 3), " (no sign, no units)")

# and the ranking is unstable when the rows change
from collections import Counter
tops = []
for b in range(30):
    idx = rng.integers(0, len(y), len(y))
    f = RandomForestRegressor(n_estimators=150, random_state=b).fit(X.iloc[idx], y.iloc[idx])
    tops.append(pd.Series(f.feature_importances_, index=X.columns).idxmax())
print("\nmost important variable across 30 resamples:", dict(Counter(tops)))

regression coefficient on weight: -0.00645
forest importance for weight:       0.676  (no sign, no units)



most important variable across 30 resamples: {'horsepower': 3, 'weight': 27}


## 5. Comparing models: two different numbers

AIC needs a likelihood. Cross-validation does not.

In [6]:
m1 = smf.ols("mpg ~ weight", data=cars).fit()
m2 = smf.ols("mpg ~ weight + horsepower", data=cars).fit()
m3 = smf.ols("mpg ~ weight + horsepower + model_year", data=cars).fit()
for name, m in [("weight", m1), ("+ horsepower", m2), ("+ model_year", m3)]:
    print(f"  {name:<14} AIC = {m.aic:8.1f}")

folds = KFold(5, shuffle=True, random_state=0)
for name, mod in [("linear", None), ("forest", forest)]:
    if mod is None:
        from sklearn.linear_model import LinearRegression
        mod = LinearRegression()
    s = -cross_val_score(mod, X, y, cv=folds, scoring="neg_mean_squared_error")
    print(f"  {name:<14} CV MSE = {s.mean():6.2f}")

  weight         AIC =   2263.9
  + horsepower   AIC =   2248.0
  + model_year   AIC =   2082.8
  linear         CV MSE =  11.88


  forest         CV MSE =   8.02


The AICs are comparable to each other. The cross-validated errors are comparable to each other. An AIC and a CV error are not comparable at all, so pick one method and use it for every candidate.

## 6. Parameters versus hyperparameters

One comes out of fitting. The other you choose, and cross-validation is how.

In [7]:
for depth in [2, 4, 6, 10, None]:
    f = RandomForestRegressor(n_estimators=200, max_depth=depth, random_state=0)
    s = -cross_val_score(f, X, y, cv=folds, scoring="neg_mean_squared_error")
    label = "unlimited" if depth is None else depth
    print(f"  max_depth = {str(label):<10} CV MSE = {s.mean():6.2f}")

  max_depth = 2          CV MSE =  14.16


  max_depth = 4          CV MSE =   8.22


  max_depth = 6          CV MSE =   7.91


  max_depth = 10         CV MSE =   8.06


  max_depth = unlimited  CV MSE =   8.09


`max_depth` is a hyperparameter: no amount of fitting discovers it. The honest way to set it is on folds you fixed before looking at the answer.